# 11.9 - Long-Running & Async Agents

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook builds the reliability machinery that lets an agent run for hours and survive a crash: checkpoint-and-resume, step memoization, the submit/poll/retrieve job pattern, idempotent side effects, event-driven backpressure, and a circuit breaker. Every mechanism is plain Python with no LLM key required, so you can run and step through the whole spine yourself.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A single reproducibility marker comment. Everything in this notebook is deterministic engineering logic - no model calls, no randomness that matters - so there is nothing to seed and nothing to install here.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A lone comment cell, not executable logic. It flags that the lesson's examples are meant to be deterministic and repeatable so the printed output matches the walkthroughs exactly on every run.

**How the code works - step by step**
- A single `#` comment noting the lesson uses seeded/deterministic values throughout.
- No statements execute, no state is created.

**In one line:** a marker that says "these results are reproducible," nothing more.

**What you'll see:** (no output - it is a comment-only cell)

## 1 - Checkpoint and resume: make a crash cost nothing

A long agent that crashes must not start over. The one idea here is to **save state after every step**, so a fresh worker can reload the last checkpoint and pick up exactly where the dead one stopped. This cell runs a 10-step job, crashes it at step 5, then hands the same `job_id` to a brand-new instance that finishes the rest with zero lost work.

In [ ]:
# DURABLE STATE - checkpoint after every step, resume from the last one on crash.
# A long agent that crashes must NOT start over. We keep the checkpoint in a dict
# (a real one is on disk / GCS / Postgres) so the whole survive-a-crash shape runs with no key.

STORE = {}                                       # job_id -> saved state (stands in for durable storage)

class DurableAgent:
    def __init__(self, job_id):
        self.job_id = job_id
        self.state = STORE.get(job_id, {"step": 0, "results": [], "status": "pending"})

    def _checkpoint(self):
        STORE[self.job_id] = dict(self.state)     # persist after EVERY step

    def run(self, tasks, crash_at=None):
        start = self.state["step"]                # resume exactly where we left off
        print(f"  job {self.job_id}: starting at step {start}/{len(tasks)}")
        for i in range(start, len(tasks)):
            if crash_at is not None and i == crash_at:
                print(f"  ** CRASH at step {i} (state already saved through {self.state['step']}) **")
                return "crashed"
            self.state["results"].append(f"done:{tasks[i]}")
            self.state["step"] = i + 1
            self.state["status"] = "running"
            self._checkpoint()
        self.state["status"] = "completed"
        self._checkpoint()
        return "completed"

tasks = [f"doc{i}" for i in range(10)]
a = DurableAgent("job-42")
a.run(tasks, crash_at=5)                          # process 5, then crash
print(f"  after crash: {STORE['job-42']['step']} steps saved, status={STORE['job-42']['status']}")

b = DurableAgent("job-42")                         # a FRESH instance reloads the checkpoint
outcome = b.run(tasks)                              # resumes at step 5, finishes the rest
print(f"  final: {b.state['step']}/{len(tasks)} steps, status={b.state['status']}, outcome={outcome}")
print("Zero lost work: the crash cost nothing because state was checkpointed every step.")
# Output:
#   job job-42: starting at step 0/10
#   ** CRASH at step 5 (state already saved through 5) **
#   after crash: 5 steps saved, status=running
#   job job-42: starting at step 5/10
#   final: 10/10 steps, status=completed, outcome=completed
# Zero lost work: the crash cost nothing because state was checkpointed every step.

**Explanation**

The `DurableAgent` class is a survive-a-crash harness built around one move: persist the full state (`step`, `results`, `status`) after each step into a store. `run` resumes from whatever step the loaded state reports, so crash recovery is just "construct a new agent with the same job_id and call run again." The in-memory `STORE` dict stands in for real durable storage (disk, GCS, Postgres) but the shape is identical.

**How the code works - step by step**
- **`STORE`** - a module-level dict mapping `job_id` to saved state; the stand-in for durable storage.
- **`__init__`** - loads existing state for the `job_id` from `STORE`, or starts fresh at `{step:0, results:[], status:'pending'}`.
- **`_checkpoint`** - writes a copy of the current state back to `STORE` after every step.
- **`run`** - starts the loop at `self.state['step']` (the resume point), appends a result and checkpoints each iteration; if `crash_at` matches, it returns `'crashed'` before doing that step's work.
- **The driver** - agent `a` processes 5 steps then crashes; agent `b` (a fresh instance, same `job_id`) reloads the checkpoint and finishes steps 5-9.

**In one line:** checkpoint after every step -> a fresh instance reloads and resumes -> the crash costs nothing.

**What you'll see:** The first agent starts at step 0/10 and crashes at step 5 with 5 steps saved (status=running); a fresh agent resumes at step 5 and reaches 10/10, status=completed, printing "Zero lost work."

## 2 - Step memoization: never re-run completed work

Durable state stops you losing progress; memoization stops you *paying for it twice*. When a workflow retries after a crash, completed steps should return a cached result and do no work - for an LLM agent that is the difference between re-spending every token and spending none. This cell runs a 5-step chain, crashes at step 3, retries, and counts how much real work actually happened.

In [ ]:
# STEP MEMOIZATION - completed steps never re-run on retry. This is why durable-execution
# frameworks (Temporal replay, Inngest steps) exist: after a crash, re-running a done step
# would re-spend its LLM tokens. Memoize the result and a retry returns it for free.

CACHE = {}                                         # step_id -> result (the memo / journal)
work_units = {"n": 0}                              # counts REAL work done (stands in for tokens)

def step(step_id, fn):
    if step_id in CACHE:                           # already completed -> return cached, do NO work
        print(f"  {step_id}: cached (0 work)")
        return CACHE[step_id]
    result = fn()                                  # first time -> actually run it
    work_units["n"] += 1
    CACHE[step_id] = result
    print(f"  {step_id}: ran (1 work unit)")
    return result

def run_chain(crash_at=None):
    for i in range(1, 6):
        if crash_at is not None and i == crash_at:
            print(f"  ** CRASH at step{i} **")
            raise RuntimeError("boom")
        step(f"step{i}", lambda i=i: f"result{i}")

print("First attempt (crashes at step 3):")
try:
    run_chain(crash_at=3)
except RuntimeError:
    pass
print(f"  work done so far: {work_units['n']} units\n")

print("Retry (no crash) - completed steps are memoized:")
run_chain()
print(f"\n  total work units: {work_units['n']} (5 steps, but only ran each ONCE)")
print("Steps 1-2 returned cached on retry: memoization saved re-spending their tokens.")
# Output:
# First attempt (crashes at step 3):
#   step1: ran (1 work unit)
#   step2: ran (1 work unit)
#   ** CRASH at step3 **
#   work done so far: 2 units
#
# Retry (no crash) - completed steps are memoized:
#   step1: cached (0 work)
#   step2: cached (0 work)
#   step3: ran (1 work unit)
#   step4: ran (1 work unit)
#   step5: ran (1 work unit)
#
#   total work units: 5 (5 steps, but only ran each ONCE)
# Steps 1-2 returned cached on retry: memoization saved re-spending their tokens.

**Explanation**

This is a token-accounting harness, not a model call. The `step` wrapper is the whole idea: check a cache before running, and if the step already completed, return its stored result and increment nothing. A `work_units` counter stands in for tokens so you can literally count what a crash-retry did and did not re-spend. This is exactly why durable-execution frameworks (Temporal replay, Inngest steps) exist for LLM workloads.

**How the code works - step by step**
- **`CACHE`** - the memo/journal mapping `step_id` to its result.
- **`work_units`** - a counter of REAL work performed, standing in for tokens spent.
- **`step(step_id, fn)`** - if the id is cached, print "cached (0 work)" and return it; otherwise run `fn`, bump `work_units`, store, and print "ran (1 work unit)."
- **`run_chain(crash_at)`** - loops steps 1-5, raising `RuntimeError` when it hits `crash_at`.
- **The driver** - first attempt crashes at step 3 (2 units done); the retry finds steps 1-2 cached and runs only 3-5.

**In one line:** cache each completed step -> a retry returns the done ones for free -> pay for each step once.

**What you'll see:** First attempt runs step1-2 then crashes at step3 (2 units); the retry prints step1-2 as cached (0 work) and runs step3-5, ending at 5 total work units - each step ran exactly once despite the crash.

## 3 - The async job pattern: submit, poll, retrieve

A request that waits eight hours is a broken request - it times out and holds a connection. The fix is to **submit** the work and get a `job_id` back instantly, let a background worker **run** it, and **poll** a status endpoint until it reports completed. This cell models that as a small state machine you can step through by hand.

In [ ]:
# THE ASYNC JOB PATTERN - submit -> poll -> retrieve. A long job must not block the request:
# the client submits and gets a job_id, the work runs in the background, and the client POLLS
# for status until it is done. We model the job as a state machine you can step through.

JOBS = {}                                          # job_id -> job record (stands in for a DB)

def submit(job_id, items):                         # POST /jobs -> returns immediately
    JOBS[job_id] = {"status": "queued", "progress": 0, "total": len(items), "items": items, "results": []}
    return {"job_id": job_id, "status": "queued"}

def tick(job_id):                                  # the background worker does ONE unit of work
    j = JOBS[job_id]
    if j["progress"] >= j["total"]:
        j["status"] = "completed"; return
    j["status"] = "running"
    j["results"].append(f"done:{j['items'][j['progress']]}")
    j["progress"] += 1
    if j["progress"] == j["total"]:
        j["status"] = "completed"

def poll(job_id):                                  # GET /jobs/{id} -> current status + progress
    j = JOBS[job_id]
    return {"status": j["status"], "progress": j["progress"], "total": j["total"]}

print("Client submits, then polls until done:")
print("  submit ->", submit("job-7", ["a", "b", "c", "d", "e"]))
polls = 0
while poll("job-7")["status"] != "completed":
    tick("job-7")                                  # (a real worker runs this off the request thread)
    s = poll("job-7")
    polls += 1
    print(f"  poll {polls}: status={s['status']} progress={s['progress']}/{s['total']}")
print(f"Done after {polls} polls; the submit call returned instantly and never blocked.")
# Output:
# Client submits, then polls until done:
#   submit -> {'job_id': 'job-7', 'status': 'queued'}
#   poll 1: status=running progress=1/5
#   poll 2: status=running progress=2/5
#   poll 3: status=running progress=3/5
#   poll 4: status=running progress=4/5
#   poll 5: status=completed progress=5/5
# Done after 5 polls; the submit call returned instantly and never blocked.

**Explanation**

Three functions model an HTTP job API over an in-memory `JOBS` dict: `submit` is the POST that returns immediately, `tick` is one unit of background work, and `poll` is the GET that reads status and progress. The point is decoupling - the submit call never blocks, and the client learns about completion by asking, not by waiting on the request thread. In production `tick` runs off-request under Cloud Tasks, Celery, or a durable workflow.

**How the code works - step by step**
- **`JOBS`** - a dict of `job_id` to job record; the stand-in for a jobs database.
- **`submit`** - creates the record as `queued` with progress 0 and returns `{job_id, status}` right away.
- **`tick`** - the background worker: advances one item, appends a result, flips status to `running`, and to `completed` on the last item.
- **`poll`** - returns the current `status`, `progress`, and `total` (the GET /jobs/{id} endpoint).
- **The driver** - submits 5 items, then loops calling `tick` + `poll` until status is `completed`, counting the polls.

**In one line:** submit returns a job_id instantly -> a worker ticks progress -> the client polls until completed.

**What you'll see:** Submit returns `{'job_id':'job-7','status':'queued'}`; five polls show progress climb 1/5 -> 5/5 with status flipping running -> completed, ending "Done after 5 polls; the submit call returned instantly and never blocked."

## 4 - Delivery semantics: at-least-once + idempotency

Distributed queues deliver **at-least-once** - a worker can finish a task then crash before acknowledging, so the queue re-delivers it. Exactly-once is provably impossible across a network boundary (the Two Generals problem), so the real pattern is at-least-once delivery + idempotent side effects = effectively-once. This cell delivers the same "charge" twice and shows the naive version double-charging while a keyed version dedupes.

In [ ]:
# DELIVERY SEMANTICS - exactly-once is impossible across a boundary (Two Generals). The real
# pattern is AT-LEAST-ONCE delivery + IDEMPOTENT side effects = effectively-once. Make the ACTION
# idempotent with a key, so a re-delivered message runs the side effect only ONCE.

charges = []                                       # the real-world side effect: money moved

def charge_naive(customer, amount):                # BUG: no idempotency - runs every delivery
    charges.append((customer, amount))
    return "charged"

seen_keys = set()
def charge_idempotent(key, customer, amount):      # FIX: an idempotency key dedupes retries
    if key in seen_keys:
        return "duplicate ignored"
    seen_keys.add(key)
    charges.append((customer, amount))
    return "charged"

# The queue delivers the SAME task twice (at-least-once is normal: a retry after a lost ack).
print("At-least-once delivery re-sends the same task. Watch the side effect:")
charges.clear()
print("  naive     delivery 1 ->", charge_naive("alice", 500))
print("  naive     delivery 2 ->", charge_naive("alice", 500))
print(f"  -> charges recorded: {len(charges)}  (BUG: alice paid twice)\n")

charges.clear(); seen_keys.clear()
key = "order-123"                                  # same key for both deliveries of the same task
print("  idempotent delivery 1 ->", charge_idempotent(key, "alice", 500))
print("  idempotent delivery 2 ->", charge_idempotent(key, "alice", 500))
print(f"  -> charges recorded: {len(charges)}  (FIXED: alice paid once)")
print("Make side effects idempotent (keys, UPSERT) - never the LLM output, which is non-deterministic.")
# Output:
# At-least-once delivery re-sends the same task. Watch the side effect:
#   naive     delivery 1 -> charged
#   naive     delivery 2 -> charged
#   -> charges recorded: 2  (BUG: alice paid twice)
#
#   idempotent delivery 1 -> charged
#   idempotent delivery 2 -> duplicate ignored
#   -> charges recorded: 1  (FIXED: alice paid once)
# Make side effects idempotent (keys, UPSERT) - never the LLM output, which is non-deterministic.

**Explanation**

A side-by-side contrast, not a queue implementation: `charge_naive` runs the side effect on every delivery, `charge_idempotent` guards it with a key. The load-bearing lesson is that idempotency lives on the ACTION (charge, email, write) via a key or UPSERT - never on the non-deterministic LLM output. The `charges` list is the real-world side effect (money moved) so you can count how many times it actually fired.

**How the code works - step by step**
- **`charges`** - a list recording each real side effect (money moved).
- **`charge_naive`** - appends a charge every call; with no key, two deliveries = two charges (the bug).
- **`seen_keys` + `charge_idempotent`** - if the idempotency key was already seen, return "duplicate ignored"; otherwise record the key and charge once.
- **The driver** - delivers the same task twice to each function; the naive path records 2 charges, the keyed path (same `order-123` key both times) records 1.

**In one line:** the same message arrives twice -> no key double-charges, an idempotency key dedupes to one.

**What you'll see:** The naive path prints two "charged" and records 2 charges (alice paid twice); the idempotent path prints "charged" then "duplicate ignored" and records 1 charge (alice paid once), closing with "Make side effects idempotent - never the LLM output."

## 5 - Event-driven triggers and backpressure

Most long agents are not called by a waiting user - they are **triggered by an event**: a file upload, a cron schedule, a webhook. A router dispatches each event to a handler, but bursts are real compute and real tokens, so you need **backpressure**: a concurrency cap that runs a fixed number at once and queues the rest. This cell routes six events through three handlers with a cap of two and watches four events wait in the queue.

In [ ]:
# EVENT-DRIVEN TRIGGERS + BACKPRESSURE - long agents usually run on an EVENT (file upload,
# cron, webhook), not a request. A router dispatches each event to the right handler; a
# concurrency CAP (backpressure) holds the overflow in a queue so a flood cannot bankrupt you.

def on_upload(e):  return f"process {e['file']}"
def on_cron(e):    return f"daily report: {e['topic']}"
def on_webhook(e): return f"respond to PR #{e['pr']}"

ROUTES = {"upload": on_upload, "cron": on_cron, "webhook": on_webhook}

events = [
    {"type": "upload",  "file": "a.pdf"},
    {"type": "upload",  "file": "b.pdf"},
    {"type": "cron",    "topic": "sales"},
    {"type": "webhook", "pr": 12},
    {"type": "upload",  "file": "c.pdf"},
    {"type": "webhook", "pr": 13},
]

MAX_CONCURRENT = 2                                  # the backpressure cap
running, queue, done = [], [], []                   # in-flight, backpressure queue, finished

def admit(e):                                       # a new event arrives
    if len(running) < MAX_CONCURRENT:
        running.append(e); return "running"
    queue.append(e); return "QUEUED (backpressure)"  # cap reached -> hold it, do not fire

def drain():                                        # a slot frees up: finish one, pull the next
    e = running.pop(0)
    done.append(ROUTES[e["type"]](e))
    if queue:
        running.append(queue.pop(0))

print(f"Admit {len(events)} events with a concurrency cap of {MAX_CONCURRENT}:")
for e in events:
    label = e.get("file") or e.get("topic") or e.get("pr")
    print(f"  {e['type']:8} {str(label):7} -> {admit(e):22} (running {len(running)}, queued {len(queue)})")
peak_queued = len(queue)
while running:                                      # drain everything to completion
    drain()
print(f"\nPeak queue depth: {peak_queued} - the cap held {peak_queued} events back instead of firing all {len(events)} at once.")
print(f"All {len(done)} handled. Backpressure is reliability AND cost control - every unthrottled LLM call is real money.")
# Output:
# Admit 6 events with a concurrency cap of 2:
#   upload   a.pdf   -> running                (running 1, queued 0)
#   upload   b.pdf   -> running                (running 2, queued 0)
#   cron     sales   -> QUEUED (backpressure)  (running 2, queued 1)
#   webhook  12      -> QUEUED (backpressure)  (running 2, queued 2)
#   upload   c.pdf   -> QUEUED (backpressure)  (running 2, queued 3)
#   webhook  13      -> QUEUED (backpressure)  (running 2, queued 4)
#
# Peak queue depth: 4 - the cap held 4 events back instead of firing all 6 at once.
# All 6 handled. Backpressure is reliability AND cost control - every unthrottled LLM call is real money.

**Explanation**

This models a trigger router plus an admission controller. `ROUTES` maps event types to handlers (the dispatch); `admit` and `drain` implement the concurrency cap that is the backpressure. The takeaway is that backpressure is both reliability and cost control - it stops a flood of events from launching a thousand simultaneous agents and a thousand-dollar bill by holding the overflow in a queue and releasing it as slots free up.

**How the code works - step by step**
- **`on_upload` / `on_cron` / `on_webhook`** - the three handlers, one per event type.
- **`ROUTES`** - maps each event `type` to its handler function.
- **`MAX_CONCURRENT` + `running`/`queue`/`done`** - the cap and the in-flight, backpressure, and finished lists.
- **`admit`** - runs an event if fewer than the cap are in flight, else appends it to the queue ("QUEUED (backpressure)").
- **`drain`** - finishes one running event and pulls the next queued one into the free slot.
- **The driver** - admits all 6 events (recording peak queue depth), then drains to completion.

**In one line:** route each event to its handler -> a concurrency cap runs a few and queues the burst -> drain smooths it into a steady stream.

**What you'll see:** With a cap of 2, the first two uploads run and the next four events print "QUEUED (backpressure)" (peak queue depth 4); all 6 are handled, closing that the cap held 4 back instead of firing all 6 at once.

## 6 - Reliability: circuit breaker, backoff and deadlines

LLM providers rate-limit, time out, and go down, and the naive response - retry immediately, forever - hammers a struggling provider and runs up the bill. A **circuit breaker** trips **open** after N failures and fast-fails instead of calling a dead API; after a cooldown it goes **half-open** to probe, and a success **closes** it. This cell runs a breaker against a provider that is down for five ticks, then recovers.

In [ ]:
# RELIABILITY: a CIRCUIT BREAKER protects a flaky LLM provider. After N failures it trips
# OPEN and fast-fails (no more hammering a down API); after a cooldown it goes HALF-OPEN to
# probe; a success CLOSES it again. Blind retries make an outage worse - the breaker stops that.

class CircuitBreaker:
    def __init__(self, fail_max=3, cooldown=2):
        self.fail_max, self.cooldown = fail_max, cooldown
        self.fails, self.state, self.since_open = 0, "closed", 0

    def call(self, fn, tick):
        if self.state == "open":
            if tick - self.since_open >= self.cooldown:
                self.state = "half-open"            # cooldown elapsed -> probe once
            else:
                return "FAST-FAIL (open)"           # refuse instantly, protect the provider
        try:
            result = fn()
            self.fails, self.state = 0, "closed"     # success -> reset
            return result
        except Exception:
            self.fails += 1
            if self.fails >= self.fail_max:
                self.state, self.since_open = "open", tick
            return "error"

# A provider that is DOWN for ticks 0-4, then recovers.
def provider(tick):
    if tick < 5:
        raise RuntimeError("provider down")
    return "ok"

cb = CircuitBreaker(fail_max=3, cooldown=2)
print("tick | breaker state -> result")
for t in range(9):
    r = cb.call(lambda t=t: provider(t), tick=t)
    print(f"  {t}  | {cb.state:9} -> {r}")
print("\nFailures tripped the breaker OPEN (fast-fail), the cooldown half-opened it, recovery CLOSED it.")
print("Pair with retry+backoff and a dead-letter queue for tasks that still exhaust their retries.")
# Output:
# tick | breaker state -> result
#   0  | closed    -> error
#   1  | closed    -> error
#   2  | open      -> error
#   3  | open      -> FAST-FAIL (open)
#   4  | open      -> error
#   5  | open      -> FAST-FAIL (open)
#   6  | closed    -> ok
#   7  | closed    -> ok
#   8  | closed    -> ok
#
# Failures tripped the breaker OPEN (fast-fail), the cooldown half-opened it, recovery CLOSED it.
# Pair with retry+backoff and a dead-letter queue for tasks that still exhaust their retries.

**Explanation**

A small state machine (closed -> open -> half-open -> closed) wrapped around a flaky call. `call` is the whole logic: fast-fail while open, probe once after the cooldown, reset on success and count failures otherwise. The point is defense in depth - the breaker is one layer that should be paired with retry+backoff (space the attempts), a dead-letter queue (catch tasks that exhaust retries), and a deadline (stop a stuck agent).

**How the code works - step by step**
- **`CircuitBreaker.__init__`** - sets `fail_max`, `cooldown`, and initial state (`closed`, 0 failures).
- **`call`** - if open and the cooldown has elapsed, go half-open to probe; if open and not, return "FAST-FAIL (open)"; otherwise run the fn, reset on success, or count the failure and trip open at `fail_max`.
- **`provider(tick)`** - raises "provider down" for ticks 0-4, returns "ok" from tick 5 on (a transient outage).
- **The driver** - calls the breaker across ticks 0-8 and prints the state and result each tick.

**In one line:** count failures -> trip open and fast-fail -> half-open probe after cooldown -> success closes it.

**What you'll see:** Ticks 0-1 error (closed), tick 2 trips open, ticks 3 and 5 return FAST-FAIL, tick 6 probes successfully and closes the breaker so ticks 6-8 return "ok" - failures tripped it open, the cooldown half-opened it, recovery closed it.

## 7 - The stack: durable execution vs queues vs serverless

Now the judgment call: match the task to the runtime family. **Durable-execution** frameworks (Temporal, Inngest, DBOS) are for stateful, long, crash-recoverable workflows with human-in-the-loop; **task queues** (Celery, Dramatiq) for high-throughput independent units; **serverless jobs** for scheduled batches. The old 15-minute wall is Lambda-only (Cloud Run Jobs run 7 days, Fargate has no hard cap), so duration rarely decides - state and crash-recovery do. This cell routes four sample tasks to the right family.

In [ ]:
# THE STACK - choose durable execution vs a task queue vs a serverless job by the task's shape.
# Durable execution (Temporal/Inngest/DBOS) for STATEFUL, long, crash-recoverable workflows.
# Task queues (Celery/Dramatiq) for HIGH-THROUGHPUT independent units. Serverless jobs for
# scheduled/event batches. The 15-min cap is Lambda-only: Cloud Run Jobs 7d, AWS Fargate, Azure Durable.

def choose(stateful, long_running, needs_hitl, high_throughput_independent):
    if stateful and (long_running or needs_hitl):
        return ("Durable execution (Temporal / Inngest / DBOS)", "replay/memoize resumes to the last step; signals for HITL")
    if high_throughput_independent:
        return ("Task queue (Celery / Dramatiq / Taskiq)", "many independent units; no mid-task checkpoint, so keep steps short")
    if long_running:
        return ("Serverless job (Cloud Run Jobs 7d / AWS Fargate / Azure Durable)", "run a batch to completion; add your own checkpointing")
    return ("Plain async worker (FastAPI BackgroundTasks)", "small, quick, best-effort background work")

cases = [
    ("8-hour invoice pipeline with resume + approval", dict(stateful=True,  long_running=True,  needs_hitl=True,  high_throughput_independent=False)),
    ("10,000 independent classification calls",        dict(stateful=False, long_running=False, needs_hitl=False, high_throughput_independent=True)),
    ("nightly batch report over a dataset",            dict(stateful=False, long_running=True,  needs_hitl=False, high_throughput_independent=False)),
    ("fire-and-forget email summary",                  dict(stateful=False, long_running=False, needs_hitl=False, high_throughput_independent=False)),
]
print("Route each task to the right runtime:")
for name, props in cases:
    approach, why = choose(**props)
    print(f"  {name:47} -> {approach}")
    print(f"  {'':47}    ({why})")
print("\nDurable execution for stateful long workflows; queues for throughput; serverless jobs for batches.")
# Output:
# Route each task to the right runtime:
#   8-hour invoice pipeline with resume + approval  -> Durable execution (Temporal / Inngest / DBOS)
#                                                      (replay/memoize resumes to the last step; signals for HITL)
#   10,000 independent classification calls         -> Task queue (Celery / Dramatiq / Taskiq)
#                                                      (many independent units; no mid-task checkpoint, so keep steps short)
#   nightly batch report over a dataset             -> Serverless job (Cloud Run Jobs 7d / AWS Fargate / Azure Durable)
#                                                      (run a batch to completion; add your own checkpointing)
#   fire-and-forget email summary                   -> Plain async worker (FastAPI BackgroundTasks)
#                                                      (small, quick, best-effort background work)
#
# Durable execution for stateful long workflows; queues for throughput; serverless jobs for batches.

**Explanation**

A decision function, not a runtime. `choose` encodes the routing rule as a short cascade of booleans about the task's shape (stateful, long-running, needs HITL, high-throughput-independent) and returns the runtime plus the reason. It is the lesson's synthesis: rather than picking by how long a job runs, you pick by whether it holds state and must recover from a crash.

**How the code works - step by step**
- **`choose`** - stateful + (long or HITL) -> durable execution; else high-throughput-independent -> task queue; else long-running -> serverless job; else a plain async worker.
- **`cases`** - four tasks paired with their shape flags: an 8-hour pipeline with resume+approval, 10k independent classifications, a nightly batch, and a fire-and-forget email.
- **The driver** - runs `choose` on each case and prints the chosen runtime and the one-line reason.

**In one line:** describe the task's shape -> the cascade routes it to durable execution, a queue, a serverless job, or a plain worker.

**What you'll see:** The four cases route respectively to Durable execution (Temporal/Inngest/DBOS), a Task queue (Celery/Dramatiq/Taskiq), a Serverless job (Cloud Run Jobs 7d/Fargate/Azure Durable), and a Plain async worker - each with a parenthetical reason.

## Setup - install dependencies

An optional dependency install for running the notebook in a fresh environment such as Colab. Every concept cell above is pure keyless Python, so this is only relevant if you extend the examples with real provider calls.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

Environment prep, commented out by default. It installs `python-dotenv` so the next cell can optionally load keys from a `.env` file; nothing here is needed by the seven keyless mechanism cells above.

**How the code works - step by step**
- A commented `!pip install python-dotenv -q` line to uncomment on Colab or a fresh environment.

**In one line:** optional install for `.env` loading; not required by any cell in this lesson.

**What you'll see:** (no output - environment prep, and it is commented out by default)

## Setup - provider API keys (optional)

Reads provider keys from the environment for anyone who wants to swap a real LLM call into the illustrative durable-execution example. The seven mechanism cells run entirely without keys, so this cell is a convenience placeholder.

> **No key needed for this lesson** - the demos above are pure Python. Set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, or `GOOGLE_API_KEY` only if you extend them with real provider calls.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Key-loading boilerplate that never hardcodes secrets. It uses `os.environ.setdefault` so any keys already present in the environment win, and empty placeholders are used otherwise; it exists for optional multi-provider extensions, not for the keyless cells above.

**How the code works - step by step**
- **`import os`** - to read and set environment variables.
- **`os.environ.setdefault(...)`** - registers `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GOOGLE_API_KEY`, defaulting to empty strings so existing env values are preserved.

**In one line:** load provider keys from the environment (any one is enough) without hardcoding, for optional real-call extensions.

**What you'll see:** (no output - environment prep)

Together these seven mechanisms are the anatomy of an agent that can run for eight hours and shrug off a crash: checkpoint every step so a restart loses nothing, memoize completed steps so a retry re-spends no tokens, decouple the work from the request with submit/poll/retrieve, make the side effects idempotent so at-least-once delivery becomes effectively-once, run on events with a concurrency cap, and wrap flaky providers in a circuit breaker with backoff and deadlines. The final router turns all of that into a single judgment call - pick the runtime by state and crash-recovery, not by duration. Next, Lesson 11.10 adds human-in-the-loop approval as a durable signal, and 11.11 covers evaluating and monitoring these long agents.